# LeetCode 518 — Coin Change II bằng Python

Link đề: https://leetcode.com/problems/coin-change-ii/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Unbounded Knapsack (cái túi không giới hạn số lần lấy)

Skeleton LeetCode:

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        ...
```

Ghi chú: ví dụ 4 (`amount = 0`) là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **đếm số tổ hợp** các đồng xu (mỗi loại dùng bao nhiêu lần tùy ý) có tổng đúng bằng `amount`; hai cách chỉ khác **thứ tự** thì tính là **một**.
2. Nắm các công cụ Python cần dùng: `itertools.product` và `combinations_with_replacement` để vét cạn, `enumerate(coins, 1)`, `functools.lru_cache`, tạo mảng `[1] + [0] * n`, và cái bẫy **chỉ số âm** trong list.
3. Đi từ **vét cạn mọi bộ số lượng** → **đệ quy "lấy/bỏ" + ghi nhớ** → **bảng DP 2 chiều** → **một mảng (cái túi không giới hạn)** → **nhân đa thức (hàm sinh)**.
4. Hiểu mấu chốt: chia các cách thành hai nhóm **không dùng đồng xu hiện tại** và **dùng ít nhất một đồng** (bỏ đi một đồng thì thành bài nhỏ hơn); và **thứ tự hai vòng lặp** quyết định ta đếm *tổ hợp* hay *hoán vị*.
5. Tránh các lỗi kinh điển: đặt vòng `amount` ở ngoài (đếm thành hoán vị), quên `f[0] = 1`, lặp ngược trong vòng `amount` (biến thành cái túi 0/1), và để chỉ số âm lọt vào (Python không báo lỗi).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `coins` là mệnh giá các loại đồng xu, và số nguyên `amount` là tổng tiền cần đạt. Hãy trả về **số tổ hợp** đồng xu tạo ra đúng `amount`. Nếu không tổ hợp nào tạo được thì trả `0`.

Mỗi loại xu có **vô hạn** đồng. Đáp án đảm bảo vừa trong số nguyên 32-bit có dấu.

**Ví dụ 1**

- Input: `amount = 5, coins = [1,2,5]`
- Output: `4`
- Giải thích: có 4 cách: `5`, `2+2+1`, `2+1+1+1`, `1+1+1+1+1`.

**Ví dụ 2**

- Input: `amount = 3, coins = [2]`
- Output: `0`
- Giải thích: chỉ dùng xu 2 thì không tạo được 3.

**Ví dụ 3**

- Input: `amount = 10, coins = [10]`
- Output: `1`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `amount = 0, coins = [7]`
- Output: `1`
- Giải thích: tổng `0` đạt được bằng cách **không lấy đồng nào** — đó là đúng 1 tổ hợp (tổ hợp rỗng). Đây là điểm dễ sai nhất ở trường hợp biên.

**Ràng buộc:**

- `1 <= coins.length <= 300`
- `1 <= coins[i] <= 5000`
- Mọi giá trị trong `coins` **khác nhau**.
- `0 <= amount <= 5000`

Chú ý: "tổ hợp" nghĩa là **không quan tâm thứ tự**: `2+2+1` và `1+2+2` chỉ là một cách.

In [1]:
from typing import List
from functools import lru_cache
from itertools import product, combinations_with_replacement
import random
import sys
import time

examples = [
    (5, [1, 2, 5], 4),
    (3, [2], 0),
    (10, [10], 1),
    (0, [7], 1),
]
for amount, coins, expected in examples:
    print(f"amount = {amount}, coins = {coins} -> kỳ vọng {expected}")

amount = 5, coins = [1, 2, 5] -> kỳ vọng 4
amount = 3, coins = [2] -> kỳ vọng 0
amount = 10, coins = [10] -> kỳ vọng 1
amount = 0, coins = [7] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tổ hợp (không thứ tự) và hoán vị (có thứ tự)

Với `amount = 3`, `coins = [1, 2]`:

- **Tổ hợp** (bài này): `{1,1,1}` và `{1,2}` — **2** cách.
- **Hoán vị / dãy có thứ tự** (bài Combination Sum IV, 377): `1+1+1`, `1+2`, `2+1` — **3** cách.

Một tổ hợp có thể viết gọn là **bộ số lượng** `(số xu loại 1, số xu loại 2, ...)`. Hai dãy khác thứ tự nhưng cùng bộ số lượng thì là cùng một tổ hợp.

In [2]:
coins = [1, 2]
amount = 3

# Tổ hợp: liệt kê "đa tập" tăng dần các mệnh giá bằng combinations_with_replacement
combos = []
for k in range(1, amount + 1):
    for c in combinations_with_replacement(coins, k):
        if sum(c) == amount:
            combos.append(c)
print("tổ hợp  :", combos)

# Hoán vị: liệt kê mọi dãy có thứ tự bằng product
perms = []
for k in range(1, amount + 1):
    for c in product(coins, repeat=k):
        if sum(c) == amount:
            perms.append(c)
print("hoán vị :", perms)

tổ hợp  : [(1, 2), (1, 1, 1)]
hoán vị : [(1, 2), (2, 1), (1, 1, 1)]


### 3.2. Mảng đếm `[1] + [0] * n`

Trong DP đếm ta thường dùng mảng `f` với `f[j]` là "số cách tạo được tổng `j`". Muốn `f[0] = 1` còn lại `0`, ta nối hai list: `[1] + [0] * n` có độ dài `n + 1`. Phép `*` lặp list; ở đây phần tử là số nguyên (bất biến) nên an toàn.

In [3]:
n = 5
f = [1] + [0] * n
print(f, "| độ dài =", len(f))

[1, 0, 0, 0, 0, 0] | độ dài = 6


### 3.3. Cái túi không giới hạn (unbounded knapsack) bằng vòng lặp

"Mỗi loại dùng vô hạn lần" nghĩa là khi đã chọn một đồng `x`, ta vẫn **được chọn tiếp chính `x`**. Trong code, điều đó thể hiện bằng việc khi tính `f[j]` ta được đọc `f[j - x]` **của cùng lượt** (đã tính xong, có thể đã dùng `x`). Hãy xem một ví dụ nhỏ nhất: chỉ có xu `2`.

In [4]:
amount = 8
f = [1] + [0] * amount
x = 2
for j in range(x, amount + 1):
    f[j] += f[j - x]
    print(f"j = {j}: f[{j}] += f[{j - x}] -> f = {f}")

j = 2: f[2] += f[0] -> f = [1, 0, 1, 0, 0, 0, 0, 0, 0]
j = 3: f[3] += f[1] -> f = [1, 0, 1, 0, 0, 0, 0, 0, 0]
j = 4: f[4] += f[2] -> f = [1, 0, 1, 0, 1, 0, 0, 0, 0]
j = 5: f[5] += f[3] -> f = [1, 0, 1, 0, 1, 0, 0, 0, 0]
j = 6: f[6] += f[4] -> f = [1, 0, 1, 0, 1, 0, 1, 0, 0]
j = 7: f[7] += f[5] -> f = [1, 0, 1, 0, 1, 0, 1, 0, 0]
j = 8: f[8] += f[6] -> f = [1, 0, 1, 0, 1, 0, 1, 0, 1]


Tổng `2, 4, 6, 8` đều có đúng **1 cách** (`2`, `2+2`, `2+2+2`, ...), tổng lẻ có `0` cách. Điều này nhờ ta đọc `f[j - 2]` đã được cập nhật trong **cùng vòng lặp** — chính là "dùng xu 2 nhiều lần".

### 3.4. `enumerate(coins, 1)`

`enumerate(iterable, start)` đánh số từ `start`. Với `coins = [1, 2, 5]` ta có cặp `(1, 1), (2, 2), (3, 5)` — chỉ số bắt đầu từ `1` rất hợp với bảng DP có hàng `0` dành cho "chưa dùng xu nào".

In [5]:
for i, x in enumerate([1, 2, 5], 1):
    print("i =", i, " mệnh giá x =", x)

i = 1  mệnh giá x = 1
i = 2  mệnh giá x = 2
i = 3  mệnh giá x = 5


### 3.5. `functools.lru_cache`

Gắn `@lru_cache(maxsize=None)` lên hàm đệ quy thuần túy để **nhớ** kết quả mỗi bộ tham số, tránh tính lại. Kiểm tra số lần thật sự tính bằng `cache_info().misses`.

In [6]:
@lru_cache(maxsize=None)
def tri(k):
    return 0 if k == 0 else k + tri(k - 1)

print(tri(100), tri.cache_info())

5050 CacheInfo(hits=0, misses=101, maxsize=None, currsize=101)


### 3.6. Bẫy chỉ số âm trong list Python

Python cho phép `a[-1]` là phần tử **cuối**. Vì vậy nếu bạn viết `f[j - x]` khi `j < x` thì **không lỗi**, nhưng đọc nhầm sang phần cuối mảng — một lỗi âm thầm rất nguy hiểm trong DP.

In [7]:
f = [10, 20, 30, 40]
j, x = 1, 3
print("j - x =", j - x, "-> f[j - x] =", f[j - x], "(thực ra là phần tử cuối f[-2])")

j - x = -2 -> f[j - x] = 30 (thực ra là phần tử cuối f[-2])


### 3.7. Đa thức biểu diễn bằng list (cần cho cách 5)

Một list `p = [p0, p1, p2, ...]` có thể xem như đa thức `p0 + p1·x + p2·x² + ...`. Nhân hai đa thức: hệ số `x^k` của tích là `sum(p[a] * q[b] for a + b = k)`. Ta thường **cắt** kết quả ở bậc `amount` vì phần cao hơn không cần.

In [8]:
def poly_mul(p, q, limit):
    r = [0] * (limit + 1)
    for a, pa in enumerate(p):
        if pa == 0:
            continue
        for b, qb in enumerate(q):
            if a + b > limit:
                break
            r[a + b] += pa * qb
    return r

# (1 + x)(1 + x) = 1 + 2x + x^2
print(poly_mul([1, 1], [1, 1], 5))
# (1 + x^2 + x^4)(1 + x + x^2 + x^3) cắt ở bậc 5
print(poly_mul([1, 0, 1, 0, 1], [1, 1, 1, 1], 5))

[1, 2, 1, 0, 0, 0]
[1, 1, 2, 2, 2, 2]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `amount = 5`, `coins = [1, 2, 5]`. Muốn **không đếm trùng** các cách chỉ khác thứ tự, ta quyết định **theo từng loại xu**, theo một thứ tự cố định: "dùng bao nhiêu xu `5`? bao nhiêu xu `2`? bao nhiêu xu `1`?".

- Dùng 1 xu `5` (còn `0`): xu 2 và 1 đều dùng 0 → `5`. **1 cách.**
- Dùng 0 xu `5` (còn `5`):
  - 2 xu `2` (còn 1) → 1 xu `1`: `2+2+1`.
  - 1 xu `2` (còn 3) → 3 xu `1`: `2+1+1+1`.
  - 0 xu `2` (còn 5) → 5 xu `1`: `1+1+1+1+1`.

Tổng: `1 + 3 = 4` ✓.

Mỗi tổ hợp tương ứng **đúng một** bộ số lượng `(a, b, c)` thỏa `5a + 2b + 1c = 5`; vì ta duyệt theo thứ tự loại xu cố định nên không bao giờ đếm một tổ hợp hai lần. Hàm dưới in từng tổ hợp.

In [9]:
def list_combos(amount, coins):
    """Liệt kê mọi tổ hợp dưới dạng {mệnh giá: số lượng}, đi theo thứ tự coins."""
    out = []
    def go(i, remain, chosen):
        if i == len(coins):
            if remain == 0:
                out.append(dict(chosen))
            return
        x = coins[i]
        for k in range(remain // x + 1):
            chosen[x] = k
            go(i + 1, remain - k * x, chosen)
        chosen.pop(x, None)
    go(0, amount, {})
    return out

combos = list_combos(5, [5, 2, 1])
for c in combos:
    print({x: k for x, k in c.items() if k > 0})
print("số tổ hợp:", len(combos))

{1: 5}
{2: 1, 1: 3}
{2: 2, 1: 1}
{5: 1}
số tổ hợp: 4


## 5. Cách 1 — Brute Force: thử mọi bộ số lượng

**Ý tưởng:** xu loại `c` dùng được `0, 1, ..., amount // c` đồng. Dùng `itertools.product` lập **mọi bộ số lượng** `(k_1, ..., k_m)` rồi đếm những bộ có `Σ k_i · c_i == amount`.

Rất dễ hiểu và đúng, nên ta dùng làm "trọng tài" (oracle) kiểm tra các cách nhanh hơn trên input nhỏ.

In [10]:
def change_bruteforce(amount: int, coins: List[int]) -> int:
    ranges = [range(amount // c + 1) for c in coins]
    count = 0
    for counts in product(*ranges):
        if sum(k * c for k, c in zip(counts, coins)) == amount:
            count += 1
    return count

for amount, coins, expected in examples:
    got = change_bruteforce(amount, coins)
    print(amount, coins, got, "OK" if got == expected else "SAI")

5 [1, 2, 5] 4 OK
3 [2] 0 OK
10 [10] 1 OK
0 [7] 1 OK


**Độ phức tạp:** số bộ số lượng là `∏ (amount // c_i + 1)`, tăng **theo cấp số nhân** với số loại xu `m`; mỗi bộ tốn `O(m)` để cộng ⇒ rất chậm.

**Nút thắt:** `m` tới 300 và `amount` tới 5000 thì số bộ là con số khổng lồ. Hãy đo với `coins = [1, 2, ..., m]`, `amount = 12`.

In [11]:
for m in [2, 4, 6, 8]:
    coins = list(range(1, m + 1))
    n_tuples = 1
    for c in coins:
        n_tuples *= 12 // c + 1
    t0 = time.perf_counter()
    r = change_bruteforce(12, coins)
    dt = time.perf_counter() - t0
    print(f"m = {m}: {n_tuples:9d} bộ số lượng, kết quả {r:3d}, ~{dt * 1000:7.1f} ms")

m = 2:        91 bộ số lượng, kết quả   7, ~    0.1 ms
m = 4:      1820 bộ số lượng, kết quả  34, ~    1.9 ms
m = 6:     16380 bộ số lượng, kết quả  58, ~   18.5 ms
m = 8:     65520 bộ số lượng, kết quả  70, ~   42.5 ms


## 6. Ý tưởng thuật toán / bước đột phá

Định nghĩa trạng thái:

> `f[i][j]` = số tổ hợp tạo được tổng `j` **chỉ dùng `i` loại xu đầu tiên** (`coins[0..i-1]`).

Xét đồng xu thứ `i` (mệnh giá `x = coins[i-1]`). Mọi tổ hợp tạo `j` chia **không giao nhau** thành hai nhóm:

1. **Không dùng loại `x`**: đó chính là các tổ hợp dùng `i - 1` loại đầu ⇒ `f[i-1][j]`.
2. **Dùng ít nhất một đồng `x`**: bỏ đi một đồng `x`, còn lại là tổ hợp tạo `j - x` mà **vẫn được dùng `i` loại xu** (kể cả `x` tiếp) ⇒ `f[i][j - x]` (chỉ xét khi `j >= x`).

Vậy:

```
f[i][j] = f[i-1][j] + f[i][j - x]   (nếu j >= x)
f[i][j] = f[i-1][j]                 (nếu j <  x)
```

Cơ sở: `f[0][0] = 1` (không có xu nào, tổng 0: một cách là tổ hợp rỗng), `f[0][j] = 0` với `j > 0`. Đáp án: `f[m][amount]`.

Tính tay với `coins = [1, 2, 5]`, `amount = 5`:

| `i \ j` | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| 0 (chưa dùng xu) | 1 | 0 | 0 | 0 | 0 | 0 |
| 1 (xu 1) | 1 | 1 | 1 | 1 | 1 | 1 |
| 2 (xu 1, 2) | 1 | 1 | 2 | 2 | 3 | 3 |
| 3 (xu 1, 2, 5) | 1 | 1 | 2 | 2 | 3 | **4** |

Ví dụ `f[2][4] = f[1][4] + f[2][2] = 1 + 2 = 3` (`1+1+1+1`, `2+1+1`, `2+2`); `f[3][5] = f[2][5] + f[3][0] = 3 + 1 = 4` ✓. Có `m · amount` ô, mỗi ô `O(1)` ⇒ `O(m · amount)`.

## 7. Các cách tiếp theo

Từ công thức trên ta có các cách cài đặt: đệ quy + nhớ, bảng 2 chiều, một mảng, và một cách có góc nhìn đại số (nhân đa thức). Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Đệ quy "lấy hoặc bỏ" + ghi nhớ (top-down memo)

**Giải thích từng đoạn:**

1. `dfs(i, remain)` = số tổ hợp tạo tổng `remain` **chỉ dùng các loại xu từ chỉ số `i` trở đi**.
2. `remain == 0`: trả `1` (đã đủ tiền, tổ hợp hoàn chỉnh).
3. `i == len(coins)` (hết loại xu mà `remain > 0`): trả `0`.
4. Ngược lại: **bỏ** loại `i` ⇒ `dfs(i + 1, remain)`; **lấy** thêm một đồng loại `i` (và vẫn được lấy tiếp) ⇒ `dfs(i, remain - coins[i])` nếu `remain >= coins[i]`.
5. `lru_cache` giữ số trạng thái tối đa `m · amount`.

Trước hết xem **không nhớ** thì số lời gọi bùng nổ ra sao.

In [12]:
def count_calls_no_memo(amount, coins):
    calls = 0
    def dfs(i, remain):
        nonlocal calls
        calls += 1
        if remain == 0:
            return 1
        if i == len(coins):
            return 0
        ways = dfs(i + 1, remain)
        if remain >= coins[i]:
            ways += dfs(i, remain - coins[i])
        return ways
    return dfs(0, amount), calls

def count_calls_memo(amount, coins):
    calls = 0
    @lru_cache(maxsize=None)
    def dfs(i, remain):
        nonlocal calls
        calls += 1                      # chỉ đếm khi thật sự tính
        if remain == 0:
            return 1
        if i == len(coins):
            return 0
        ways = dfs(i + 1, remain)
        if remain >= coins[i]:
            ways += dfs(i, remain - coins[i])
        return ways
    return dfs(0, amount), calls

for amount in [10, 20, 30, 40]:
    coins = [1, 2, 5]
    a, c1 = count_calls_no_memo(amount, coins)
    b, c2 = count_calls_memo(amount, coins)
    assert a == b
    print(f"amount = {amount:2d}, coins = {coins}: kết quả {a:3d} | không nhớ {c1:7d} lời gọi | có nhớ {c2:4d} lần tính")

amount = 10, coins = [1, 2, 5]: kết quả  10 | không nhớ     128 lời gọi | có nhớ   43 lần tính
amount = 20, coins = [1, 2, 5]: kết quả  29 | không nhớ     585 lời gọi | có nhớ   83 lần tính
amount = 30, coins = [1, 2, 5]: kết quả  58 | không nhớ    1572 lời gọi | có nhớ  123 lần tính
amount = 40, coins = [1, 2, 5]: kết quả  97 | không nhớ    3289 lời gọi | có nhớ  163 lần tính


In [13]:
class SolutionMemo:
    def change(self, amount: int, coins: List[int]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))
        m = len(coins)

        @lru_cache(maxsize=None)
        def dfs(i: int, remain: int) -> int:
            if remain == 0:
                return 1
            if i == m:
                return 0
            ways = dfs(i + 1, remain)
            if remain >= coins[i]:
                ways += dfs(i, remain - coins[i])
            return ways

        return dfs(0, amount)

for amount, coins, expected in examples:
    print(amount, coins, SolutionMemo().change(amount, coins), "kỳ vọng", expected)

5 [1, 2, 5] 4 kỳ vọng 4
3 [2] 0 kỳ vọng 0
10 [10] 1 kỳ vọng 1
0 [7] 1 kỳ vọng 1


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        m = len(coins)

        @cache
        def dfs(i: int, remain: int) -> int:
            if remain == 0:
                return 1
            if i == m:
                return 0
            ways = dfs(i + 1, remain)
            if remain >= coins[i]:
                ways += dfs(i, remain - coins[i])
            return ways

        return dfs(0, amount)
```

**Độ phức tạp:** `O(m · amount)` thời gian và bộ nhớ cho cache. Độ sâu đệ quy có thể tới `amount / min(coins) + m` (khoảng 5300), vượt ngưỡng mặc định của Python, nên trên LeetCode đôi khi cần `sys.setrecursionlimit` hoặc chuyển sang bottom-up.

### 7.2. Cách 3 — Bảng DP 2 chiều bottom-up

**Giải thích từng đoạn:**

1. `f` có `m + 1` hàng, `amount + 1` cột, toàn `0`; đặt `f[0][0] = 1`.
2. Với mỗi loại xu `x` (đánh số `i` từ 1): với mọi `j` từ `0` đến `amount`, đặt `f[i][j] = f[i-1][j]` (nhóm "không dùng `x`").
3. Nếu `j >= x` thì cộng thêm `f[i][j - x]` (nhóm "dùng ít nhất một đồng `x`"). **Điều kiện `j >= x` bắt buộc** để không đọc chỉ số âm.
4. Đáp án `f[m][amount]`.

In [14]:
class SolutionTable:
    def change(self, amount: int, coins: List[int]) -> int:
        m, n = len(coins), amount
        f = [[0] * (n + 1) for _ in range(m + 1)]
        f[0][0] = 1
        for i, x in enumerate(coins, 1):
            for j in range(n + 1):
                f[i][j] = f[i - 1][j]
                if j >= x:
                    f[i][j] += f[i][j - x]
        return f[m][n]

for amount, coins, expected in examples:
    print(amount, coins, SolutionTable().change(amount, coins), "kỳ vọng", expected)

5 [1, 2, 5] 4 kỳ vọng 4
3 [2] 0 kỳ vọng 0
10 [10] 1 kỳ vọng 1
0 [7] 1 kỳ vọng 1


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        m, n = len(coins), amount
        f = [[0] * (n + 1) for _ in range(m + 1)]
        f[0][0] = 1
        for i, x in enumerate(coins, 1):
            for j in range(n + 1):
                f[i][j] = f[i - 1][j]
                if j >= x:
                    f[i][j] += f[i][j - x]
        return f[m][n]
```

**Độ phức tạp:** thời gian `O(m · amount)`, bộ nhớ `O(m · amount)` (tối đa khoảng `300 · 5001 ≈ 1,5 triệu` ô).

### 7.3. Cách 4 — Một mảng: cái túi không giới hạn

**Quan sát:** hàng `i` chỉ đọc hàng `i - 1` (cùng cột `j`) và **chính hàng `i`** (cột `j - x`). Nếu dùng **một mảng `f`** mô tả "hàng hiện tại":

- Trước khi cập nhật, `f[j]` vẫn đang giữ `f[i-1][j]` — chính là số hạng thứ nhất, nên chỉ cần `f[j] += ...`.
- Số hạng thứ hai `f[i][j - x]` là `f[j - x]` **đã được cập nhật trong lượt này** — điều này đúng khi ta duyệt `j` **tăng dần**.

Vậy chỉ cần:

```
f = [1, 0, 0, ..., 0]
for x in coins:                 # vòng ngoài: LOẠI XU
    for j in range(x, amount + 1):   # vòng trong: TỔNG, TĂNG dần
        f[j] += f[j - x]
```

Vì `j` bắt đầu từ `x`, ta không cần kiểm `j >= x` nữa. Mảng `f` thay cho cả bảng nên bộ nhớ còn `O(amount)`.

In [15]:
class SolutionRolling:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for x in coins:
            for j in range(x, amount + 1):
                f[j] += f[j - x]
        return f[amount]

for amount, coins, expected in examples:
    print(amount, coins, SolutionRolling().change(amount, coins), "kỳ vọng", expected)

5 [1, 2, 5] 4 kỳ vọng 4
3 [2] 0 kỳ vọng 0
10 [10] 1 kỳ vọng 1
0 [7] 1 kỳ vọng 1


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for x in coins:
            for j in range(x, amount + 1):
                f[j] += f[j - x]
        return f[amount]
```

**Độ phức tạp:** thời gian `O(m · amount)`, bộ nhớ **`O(amount)`**.

### 7.4. Cách 5 — Nhân đa thức (hàm sinh)

**Góc nhìn khác:** xu loại `c` có thể dùng `0, 1, 2, ...` lần, đóng góp tổng `0, c, 2c, ...`. Gói lại thành đa thức

```
P_c(x) = 1 + x^c + x^(2c) + x^(3c) + ...
```

Nhân các `P_c(x)` của mọi loại xu: hệ số của `x^amount` trong tích đúng bằng **số cách chọn số lượng mỗi loại** sao cho tổng bằng `amount` — tức đáp án. Ta nhân dần từng thừa số (cắt ở bậc `amount`), với `poly_mul` đã học ở mục 3.7. Cách này không dùng bảng "dùng/bỏ" mà dùng phép nhân đa thức tường minh, nên là kiểm tra chéo rất độc lập (và cũng chậm hơn: `O(m · amount · amount / c)`).

In [16]:
class SolutionPoly:
    def change(self, amount: int, coins: List[int]) -> int:
        poly = [1] + [0] * amount                        # đa thức "1"
        for c in coins:
            new = [0] * (amount + 1)
            for a, pa in enumerate(poly):                # nhân poly với (1 + x^c + x^2c + ...), cắt ở bậc amount
                if pa == 0:
                    continue
                for b in range(0, amount - a + 1, c):    # b = 0, c, 2c, ...
                    new[a + b] += pa
            poly = new
        return poly[amount]

for amount, coins, expected in examples:
    print(amount, coins, SolutionPoly().change(amount, coins), "kỳ vọng", expected)

5 [1, 2, 5] 4 kỳ vọng 4
3 [2] 0 kỳ vọng 0
10 [10] 1 kỳ vọng 1
0 [7] 1 kỳ vọng 1


**Code nộp LeetCode (Cách 5)** — dùng nhân đa thức, chậm hơn nhưng với `m · amount²/c` có thể nặng ở cỡ lớn (ví dụ `coins = [1,...]`, `amount = 5000`), nên chủ yếu để học và kiểm chứng:

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        poly = [1] + [0] * amount
        for c in coins:
            new = [0] * (amount + 1)
            for a, pa in enumerate(poly):
                if pa == 0:
                    continue
                for b in range(0, amount - a + 1, c):
                    new[a + b] += pa
            poly = new
        return poly[amount]
```

**Độ phức tạp:** thời gian `O(Σ amount · amount / c_i)`, trường hợp xấu cỡ `O(m · amount²)`; bộ nhớ `O(amount)`.

Liên hệ: cách 4 chính là phiên bản **nhanh** của cách 5 — nhân với `1/(1 - x^c)` tương đương `f[j] += f[j - c]` thay vì cộng cả một chuỗi.

## 8. TRACE từng bước

Ta theo dõi cách 4 (một mảng) với `amount = 5`, `coins = [1, 2, 5]`. Bảng trạng thái sau **mỗi loại xu** (dòng cuối cùng là đáp án `f[5] = 4`):

| sau khi xử lý | `f[0]` | `f[1]` | `f[2]` | `f[3]` | `f[4]` | `f[5]` |
|---|---|---|---|---|---|---|
| khởi tạo | 1 | 0 | 0 | 0 | 0 | 0 |
| xu 1 | 1 | 1 | 1 | 1 | 1 | 1 |
| xu 2 | 1 | 1 | 2 | 2 | 3 | 3 |
| xu 5 | 1 | 1 | 2 | 2 | 3 | **4** |

Hàm dưới in **từng bước `j`** của mỗi vòng.

In [17]:
def trace_rolling(amount, coins):
    f = [1] + [0] * amount
    print("khởi tạo:", f)
    for x in coins:
        print(f"=== xử lý xu {x} ===")
        for j in range(x, amount + 1):
            before = f[j]
            f[j] += f[j - x]
            print(f"  j = {j}: f[{j}] = {before} + f[{j - x}] ({f[j - x]}) = {f[j]}   -> {f}")
        print(f"sau xu {x}:", f)
    return f[amount]

print("ĐÁP ÁN:", trace_rolling(5, [1, 2, 5]))

khởi tạo: [1, 0, 0, 0, 0, 0]
=== xử lý xu 1 ===
  j = 1: f[1] = 0 + f[0] (1) = 1   -> [1, 1, 0, 0, 0, 0]
  j = 2: f[2] = 0 + f[1] (1) = 1   -> [1, 1, 1, 0, 0, 0]
  j = 3: f[3] = 0 + f[2] (1) = 1   -> [1, 1, 1, 1, 0, 0]
  j = 4: f[4] = 0 + f[3] (1) = 1   -> [1, 1, 1, 1, 1, 0]
  j = 5: f[5] = 0 + f[4] (1) = 1   -> [1, 1, 1, 1, 1, 1]
sau xu 1: [1, 1, 1, 1, 1, 1]
=== xử lý xu 2 ===
  j = 2: f[2] = 1 + f[0] (1) = 2   -> [1, 1, 2, 1, 1, 1]
  j = 3: f[3] = 1 + f[1] (1) = 2   -> [1, 1, 2, 2, 1, 1]
  j = 4: f[4] = 1 + f[2] (2) = 3   -> [1, 1, 2, 2, 3, 1]
  j = 5: f[5] = 1 + f[3] (2) = 3   -> [1, 1, 2, 2, 3, 3]
sau xu 2: [1, 1, 2, 2, 3, 3]
=== xử lý xu 5 ===
  j = 5: f[5] = 3 + f[0] (1) = 4   -> [1, 1, 2, 2, 3, 4]
sau xu 5: [1, 1, 2, 2, 3, 4]
ĐÁP ÁN: 4


Để ý với xu `1`: `f[1] = 0 + f[0]`, rồi `f[2] = 0 + f[1]` (đã là giá trị mới = 1) ... tức "xu 1 dùng lặp lại nhiều lần" được thể hiện bằng việc đọc `f[j - 1]` **vừa cập nhật**.

Giờ trace cách 3 (bảng 2 chiều) để thấy hai nhóm "bỏ xu" và "dùng xu" tách bạch.

In [18]:
def trace_table(amount, coins):
    m = len(coins)
    f = [[0] * (amount + 1) for _ in range(m + 1)]
    f[0][0] = 1
    print("hàng 0:", f[0])
    for i, x in enumerate(coins, 1):
        for j in range(amount + 1):
            skip = f[i - 1][j]
            use = f[i][j - x] if j >= x else 0
            f[i][j] = skip + use
            note = f"bỏ xu {x}: {skip}" + (f" + dùng xu {x}: f[{i}][{j - x}] = {use}" if j >= x else " (j < x, không dùng được)")
            print(f"  f[{i}][{j}] = {f[i][j]}   ({note})")
        print(f"hàng {i} (xu {x}):", f[i])
    return f[m][amount]

print("ĐÁP ÁN:", trace_table(5, [1, 2, 5]))

hàng 0: [1, 0, 0, 0, 0, 0]
  f[1][0] = 1   (bỏ xu 1: 1 (j < x, không dùng được))
  f[1][1] = 1   (bỏ xu 1: 0 + dùng xu 1: f[1][0] = 1)
  f[1][2] = 1   (bỏ xu 1: 0 + dùng xu 1: f[1][1] = 1)
  f[1][3] = 1   (bỏ xu 1: 0 + dùng xu 1: f[1][2] = 1)
  f[1][4] = 1   (bỏ xu 1: 0 + dùng xu 1: f[1][3] = 1)
  f[1][5] = 1   (bỏ xu 1: 0 + dùng xu 1: f[1][4] = 1)
hàng 1 (xu 1): [1, 1, 1, 1, 1, 1]
  f[2][0] = 1   (bỏ xu 2: 1 (j < x, không dùng được))
  f[2][1] = 1   (bỏ xu 2: 1 (j < x, không dùng được))
  f[2][2] = 2   (bỏ xu 2: 1 + dùng xu 2: f[2][0] = 1)
  f[2][3] = 2   (bỏ xu 2: 1 + dùng xu 2: f[2][1] = 1)
  f[2][4] = 3   (bỏ xu 2: 1 + dùng xu 2: f[2][2] = 2)
  f[2][5] = 3   (bỏ xu 2: 1 + dùng xu 2: f[2][3] = 2)
hàng 2 (xu 2): [1, 1, 2, 2, 3, 3]
  f[3][0] = 1   (bỏ xu 5: 1 (j < x, không dùng được))
  f[3][1] = 1   (bỏ xu 5: 1 (j < x, không dùng được))
  f[3][2] = 2   (bỏ xu 5: 2 (j < x, không dùng được))
  f[3][3] = 2   (bỏ xu 5: 2 (j < x, không dùng được))
  f[3][4] = 3   (bỏ xu 5: 3 (j < x, không

## 9. Code nộp LeetCode

Ta chọn **Cách 4 (một mảng)**: ngắn nhất, nhanh nhất, ít bộ nhớ nhất. Đây là class `Solution` đúng chữ ký:

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for x in coins:
            for j in range(x, amount + 1):
                f[j] += f[j - x]
        return f[amount]
```

In [19]:
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for x in coins:
            for j in range(x, amount + 1):
                f[j] += f[j - x]
        return f[amount]

for amount, coins, expected in examples:
    got = Solution().change(amount, coins)
    assert got == expected, (amount, coins, got, expected)
    print(amount, coins, "->", got, "OK")

5 [1, 2, 5] -> 4 OK
3 [2] -> 0 OK
10 [10] -> 1 OK
0 [7] -> 1 OK


## 10. Vì sao đúng

**Hai nhóm không giao nhau và phủ hết.** Xét một tổ hợp tạo `j` chỉ dùng `i` loại xu đầu, và xu cuối cùng `x = coins[i-1]`:

- Nếu tổ hợp **không có** xu `x`: nó là tổ hợp của `i - 1` loại đầu ⇒ thuộc `f[i-1][j]`.
- Nếu tổ hợp **có ít nhất một** xu `x`: bỏ đúng một đồng `x` ta được một tổ hợp tạo `j - x` (có thể vẫn còn xu `x`) thuộc `f[i][j - x]`. Ngược lại từ tổ hợp thuộc `f[i][j - x]` thêm một đồng `x` ta được tổ hợp thuộc nhóm này. Phép "bỏ/thêm một đồng `x`" là **song ánh**, nên số phần tử hai bên bằng nhau.

Vì "có `x`" và "không có `x`" loại trừ nhau và bao trùm mọi khả năng nên `f[i][j] = f[i-1][j] + f[i][j - x]`.

**Vì sao không đếm trùng hoán vị.** Tổ hợp là một **đa tập** (bộ số lượng). Công thức quyết định theo **từng loại xu** theo thứ tự cố định, nên mỗi bộ số lượng chỉ được sinh ra **đúng một lần**; ta không bao giờ phân biệt "xu 2 rồi xu 1" với "xu 1 rồi xu 2".

**Vì sao nén một mảng đúng.** Khi duyệt `j` tăng dần trong lượt xu `x`: tại thời điểm cập nhật `f[j]`, ô này vẫn giữ giá trị lượt trước (= `f[i-1][j]`), còn `f[j - x]` (với `j - x < j`) đã được cập nhật ở lượt này (= `f[i][j - x]`). Đó đúng là hai số hạng cần thiết.

**Kiểm chứng bằng code:** so bảng 2 chiều với bảng nhận được bằng cách đếm trực tiếp bộ số lượng chỉ dùng `i` loại xu đầu, cho mọi `(i, j)`.

In [20]:
def count_first_i(coins_prefix, j):
    return change_bruteforce(j, coins_prefix)

coins = [2, 3, 7]
amount = 14
m = len(coins)
f = [[0] * (amount + 1) for _ in range(m + 1)]
f[0][0] = 1
for i, x in enumerate(coins, 1):
    for j in range(amount + 1):
        f[i][j] = f[i - 1][j] + (f[i][j - x] if j >= x else 0)

cells = 0
for i in range(m + 1):
    for j in range(amount + 1):
        if i == 0:
            expected = 1 if j == 0 else 0
        else:
            expected = count_first_i(coins[:i], j)
        assert f[i][j] == expected, (i, j, f[i][j], expected)
        cells += 1
print("đã kiểm", cells, "ô: f[i][j] = số tổ hợp dùng i loại xu đầu, khớp vét cạn")
for row in f:
    print(row)

đã kiểm 60 ô: f[i][j] = số tổ hợp dùng i loại xu đầu, khớp vét cạn
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
[1, 0, 1, 1, 1, 1, 2, 1, 2, 2, 2, 2, 3, 2, 3]
[1, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 5]


## 11. Time / Space Complexity

Gọi `m = len(coins)`, `n = amount`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force (product) | `O(m · ∏(n/c_i + 1))` — mũ | `O(m)` |
| 2. Memo top-down | `O(m · n)` | `O(m · n)` + ngăn xếp tới `O(n / min c + m)` |
| 3. Bảng 2 chiều | `O(m · n)` | `O(m · n)` |
| 4. Một mảng | `O(m · n)` | `O(n)` |
| 5. Nhân đa thức | `O(Σ n² / c_i)` | `O(n)` |

Ở giới hạn lớn nhất `m = 300`, `n = 5000`: cách 4 làm khoảng `Σ (n - c_i + 1)` phép cộng, tối đa cỡ `1,5 triệu` — thoải mái trong dưới một giây. Ta đếm trực tiếp.

In [21]:
def count_updates(amount, coins):
    c = 0
    for x in coins:
        for j in range(x, amount + 1):
            c += 1
    return c

print("coins = [1..300], amount = 5000 ->", count_updates(5000, list(range(1, 301))), "phép cập nhật")
print("coins = [1, 2, 5],  amount = 5000 ->", count_updates(5000, [1, 2, 5]), "phép cập nhật")
print("giới hạn trên m * (n+1) =", 300 * 5001)

coins = [1..300], amount = 5000 -> 1455150 phép cập nhật
coins = [1, 2, 5],  amount = 5000 -> 14995 phép cập nhật
giới hạn trên m * (n+1) = 1500300


## 12. Bộ test

Gom các cách vào một dict. Cách 1 và cách 5 chỉ chạy trên input nhỏ; các cách DP chạy mọi cỡ. Các phần: ví dụ đề, edge case, giá trị kinh điển đã biết, phản ví dụ (hoán vị), đối chiếu vét cạn, ngẫu nhiên có seed, và kiểm tra tốc độ ở ràng buộc lớn nhất.

In [22]:
methods_dp = {
    "memo": lambda a, c: SolutionMemo().change(a, c),
    "table": lambda a, c: SolutionTable().change(a, c),
    "rolling": lambda a, c: SolutionRolling().change(a, c),
    "final": lambda a, c: Solution().change(a, c),
}
methods_small = dict(methods_dp)
methods_small["poly"] = lambda a, c: SolutionPoly().change(a, c)

def check_all(amount, coins, expected=None, small=True):
    ms = methods_small if small else methods_dp
    results = {name: fn(amount, coins) for name, fn in ms.items()}
    assert len(set(results.values())) == 1, (amount, coins, results)
    r = results["final"]
    if expected is not None:
        assert r == expected, (amount, coins, r, expected)
    return r

# 1) Ví dụ đề
for amount, coins, expected in examples:
    check_all(amount, coins, expected)
print("ví dụ đề: OK")

# 2) Edge case
edge = [
    (0, [1], 1),                   # amount = 0: tổ hợp rỗng
    (0, [5, 7, 9], 1),
    (1, [2], 0),                   # xu lớn hơn amount
    (4, [5, 6], 0),
    (7, [7], 1),
    (6, [1], 1),                   # chỉ xu 1: luôn đúng 1 cách
    (6, [2], 1),
    (7, [2], 0),
    (10, [1, 2], 6),               # xu 1 và 2: amount // 2 + 1 cách
    (11, [1, 2], 6),
]
for amount, coins, expected in edge:
    check_all(amount, coins, expected)
print("edge case: OK")

# 3) Giá trị kinh điển đã biết (bài toán đổi tiền cổ điển)
assert check_all(100, [1, 5, 10, 25, 50], 292, small=False) == 292          # đổi 1 đô bằng xu, 5, 10, 25, 50 cent
assert check_all(200, [1, 2, 5, 10, 20, 50, 100, 200], 73682, small=False) == 73682   # Project Euler 31
print("giá trị kinh điển (292 và 73682): OK")

ví dụ đề: OK
edge case: OK
giá trị kinh điển (292 và 73682): OK


In [23]:
# 4) Phản ví dụ: nếu đếm HOÁN VỊ thì ra con số khác
def count_permutations(amount, coins):
    g = [1] + [0] * amount
    for j in range(1, amount + 1):
        for x in coins:
            if j >= x:
                g[j] += g[j - x]
    return g[amount]

for amount, coins in [(3, [1, 2]), (4, [1, 2, 3]), (5, [1, 2, 5])]:
    print(f"amount = {amount}, coins = {coins}: tổ hợp = {check_all(amount, coins)} | hoán vị (SAI cho bài này) = {count_permutations(amount, coins)}")
assert check_all(3, [1, 2]) == 2 and count_permutations(3, [1, 2]) == 3

amount = 3, coins = [1, 2]: tổ hợp = 2 | hoán vị (SAI cho bài này) = 3
amount = 4, coins = [1, 2, 3]: tổ hợp = 4 | hoán vị (SAI cho bài này) = 7
amount = 5, coins = [1, 2, 5]: tổ hợp = 4 | hoán vị (SAI cho bài này) = 9


In [24]:
# 5) Đối chiếu vét cạn: mọi amount 0..25 với nhiều bộ coins nhỏ
all_sets = [[1], [2], [3], [1, 2], [2, 3], [1, 5], [2, 5, 7], [1, 3, 4], [3, 5, 8], [1, 2, 5, 10]]
count = 0
for coins in all_sets:
    for amount in range(0, 26):
        r = check_all(amount, coins)
        assert r == change_bruteforce(amount, coins), (amount, coins)
        count += 1
print("đã đối chiếu", count, "cặp (amount, coins) với brute force: OK")

# 6) Ngẫu nhiên có seed, đúng ràng buộc (coins khác nhau)
rng = random.Random(518)
for _ in range(300):
    m = rng.randint(1, 4)
    coins = rng.sample(range(1, 12), m)
    amount = rng.randint(0, 20)
    assert check_all(amount, coins) == change_bruteforce(amount, coins), (amount, coins)
print("ngẫu nhiên nhỏ so brute force: OK")

for _ in range(40):
    m = rng.randint(1, 12)
    coins = rng.sample(range(1, 200), m)
    amount = rng.randint(0, 300)
    check_all(amount, coins)                      # đối chiếu 5 cách với nhau
print("ngẫu nhiên vừa (5 cách khớp nhau): OK")

đã đối chiếu 260 cặp (amount, coins) với brute force: OK


ngẫu nhiên nhỏ so brute force: OK
ngẫu nhiên vừa (5 cách khớp nhau): OK


In [25]:
# 7) Giới hạn lớn nhất: m = 300, amount = 5000. (Đáp án có thể vượt 32-bit khi input tự sinh; đề đảm bảo khi nộp thì không.)
big_cases = {
    "coins = 1..300": (5000, list(range(1, 301))),
    "300 mệnh giá ngẫu nhiên": (5000, rng.sample(range(1, 5001), 300)),
    "coins = [1]": (5000, [1]),
    "coins = [5000]": (5000, [5000]),
}
for name, (amount, coins) in big_cases.items():
    t0 = time.perf_counter()
    r1 = Solution().change(amount, coins)
    dt = time.perf_counter() - t0
    r2 = SolutionTable().change(amount, coins)
    assert r1 == r2
    print(f"{name:26s} -> {r1}  ({dt:.2f}s)")
assert Solution().change(5000, [1]) == 1 and Solution().change(5000, [5000]) == 1

coins = 1..300             -> 134697872234317751828184783748352530253612044833208679288948519592195681479  (0.11s)
300 mệnh giá ngẫu nhiên    -> 229043559091323412  (0.05s)
coins = [1]                -> 1  (0.00s)
coins = [5000]             -> 1  (0.00s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đặt vòng `amount` ở ngoài (đếm hoán vị thay vì tổ hợp)

Đổi thứ tự hai vòng lặp trông vô hại nhưng đổi hẳn ý nghĩa bài toán.

In [26]:
class WrongOrder:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for j in range(1, amount + 1):          # SAI: vòng tổng ở ngoài
            for x in coins:
                if j >= x:
                    f[j] += f[j - x]
        return f[amount]

print("sai :", WrongOrder().change(5, [1, 2, 5]), "(đúng phải là 4)")
print("đúng:", Solution().change(5, [1, 2, 5]))

sai : 9 (đúng phải là 4)
đúng: 4


**Vì sao sai:** khi `j` ở vòng ngoài, tại mỗi tổng ta thử **mọi loại xu cho đồng cuối cùng**, nên `1+2` và `2+1` được đếm là hai cách (đó là bài *Combination Sum IV*). **Cách sửa:** vòng **loại xu** ở ngoài, vòng **tổng** ở trong — mỗi loại xu được "quyết định xong" rồi mới sang loại kế tiếp.

### Lỗi 2: quên `f[0] = 1`

In [27]:
class WrongBase:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [0] * (amount + 1)                  # SAI: quên f[0] = 1
        for x in coins:
            for j in range(x, amount + 1):
                f[j] += f[j - x]
        return f[amount]

print("sai :", WrongBase().change(5, [1, 2, 5]), "(đúng phải là 4)")
print("đúng:", Solution().change(5, [1, 2, 5]))

sai : 0 (đúng phải là 4)
đúng: 4


**Vì sao sai:** mọi `f[j]` đều được xây từ `f[0]`; nếu `f[0] = 0` thì mọi thứ cộng lại vẫn là `0`. `f[0] = 1` nghĩa là "một cách tạo tổng 0: không lấy đồng nào" — cũng chính là lý do `amount = 0` cho đáp án `1`. **Cách sửa:** `f = [1] + [0] * amount`.

### Lỗi 3: lặp `j` giảm dần (biến thành cái túi 0/1)

In [28]:
class WrongDescending:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for x in coins:
            for j in range(amount, x - 1, -1):  # SAI: duyệt giảm dần
                f[j] += f[j - x]
        return f[amount]

print("sai :", WrongDescending().change(5, [1, 2, 5]), "(mỗi loại xu chỉ dùng tối đa 1 đồng)")
print("đúng:", Solution().change(5, [1, 2, 5]))

sai : 1 (mỗi loại xu chỉ dùng tối đa 1 đồng)
đúng: 4


**Vì sao sai:** duyệt giảm dần thì `f[j - x]` còn là giá trị **trước** lượt xu `x`, nên xu `x` chỉ được dùng **một lần** (cái túi 0/1, giống bài Subset Sum đếm). Bài này cho dùng vô hạn nên phải duyệt **tăng dần**.

### Lỗi 4: để vòng `j` bắt đầu từ 0 mà không kiểm `j >= x` (chỉ số âm lọt vào)

In [29]:
class WrongNegativeIndex:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount
        for x in coins:
            for j in range(amount + 1):               # SAI: j bắt đầu từ 0, không kiểm j >= x
                f[j] += f[j - x]                      # j < x -> chỉ số âm, đọc nhầm cuối mảng
        return f[amount]

print("sai :", WrongNegativeIndex().change(5, [1, 2, 5]), "(đúng phải là 4)")
print("đúng:", Solution().change(5, [1, 2, 5]))

sai : 8 (đúng phải là 4)
đúng: 4


**Vì sao sai:** khi `j < x`, `f[j - x]` có chỉ số âm; Python không báo lỗi mà đọc từ **cuối** mảng — nơi đã chứa số cách của các tổng lớn (từ các lượt xu trước) — rồi cộng nhầm vào `f[j]`. (Với bản 2 chiều, bug này thường "may mắn" vô hại vì cuối hàng hiện tại còn `0`, hoặc báo `IndexError` khi `x > amount + 1` — càng dễ gây nhầm lẫn.) **Cách sửa:** `if j >= x:` trước khi cộng (bản 2 chiều), hoặc duyệt `j` từ `x` như bản một mảng.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra cái túi không giới hạn đếm tổ hợp:**

- "Đếm số **cách / tổ hợp** để đạt tổng `T`", mỗi phần tử dùng **nhiều lần** ⇒ `f[j] += f[j - x]` duyệt `j` **tăng dần**.
- Hỏi "**số tổ hợp**" thì **vòng phần tử ở ngoài**; hỏi "**số dãy có thứ tự**" thì **vòng tổng ở ngoài**.
- Ràng buộc `amount ≤ 5000`, `coins ≤ 300` ⇒ khoảng `1,5 triệu` phép tính, cái túi `O(m · amount)` là vừa đủ.

**Quy tắc "hai vòng lặp" cần nhớ:**

| Mục tiêu | Vòng ngoài | Vòng trong | Hướng của `j` |
|---|---|---|---|
| Đếm **tổ hợp**, dùng vô hạn (bài này) | phần tử | tổng | tăng dần |
| Đếm **hoán vị**, dùng vô hạn (377) | tổng | phần tử | tăng dần |
| Đếm tổ hợp, mỗi phần tử **một lần** (0/1) | phần tử | tổng | **giảm dần** |
| Tối thiểu số đồng (322) | phần tử hoặc tổng | còn lại | tăng dần (`min` thay `+`) |

**Mẹo kiểm tra nhanh:** `amount = 0` luôn trả `1`; nếu có `1` trong `coins` thì đáp án luôn `>= 1`; tập `coins` chỉ có một xu `c` thì đáp án là `1` nếu `c` chia hết `amount`, ngược lại `0`.

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 322. Coin Change | số đồng ít nhất (đổi `+` thành `min`) |
| 377. Combination Sum IV | đếm hoán vị (đảo vòng lặp) |
| 279. Perfect Squares | túi không giới hạn, đổi tiền bằng số chính phương |
| 39. Combination Sum | liệt kê mọi tổ hợp (backtracking) |
| 494. Target Sum / 416. Partition Equal Subset Sum | túi 0/1 |

## 15. Biến thể

### 15.1. Số đồng ít nhất (LeetCode 322)

Cùng khung cái túi nhưng thay "cộng số cách" bằng "lấy min số đồng": `f[j] = min(f[j], f[j - x] + 1)`, `f[0] = 0`, còn lại `+inf`.

In [30]:
def coin_change_min(amount, coins):
    INF = float("inf")
    f = [0] + [INF] * amount
    for x in coins:
        for j in range(x, amount + 1):
            f[j] = min(f[j], f[j - x] + 1)
    return f[amount] if f[amount] != INF else -1

print(coin_change_min(11, [1, 2, 5]))      # 3 (5+5+1)
print(coin_change_min(3, [2]))             # -1
print(coin_change_min(0, [1]))             # 0

3
-1
0


### 15.2. Đếm dãy có thứ tự (hoán vị) — LeetCode 377

Đảo hai vòng lặp. Ta đã dùng ở mục 12 để minh họa lỗi; ở đây kiểm bằng vét cạn: đếm mọi dãy có tổng bằng `amount`.

In [31]:
def permutations_bruteforce(amount, coins):
    def go(remain):
        if remain == 0:
            return 1
        return sum(go(remain - x) for x in coins if x <= remain)
    return go(amount)

for amount, coins in [(3, [1, 2]), (4, [1, 2, 3]), (7, [2, 3])]:
    a = count_permutations(amount, coins)
    assert a == permutations_bruteforce(amount, coins)
    print(f"amount = {amount}, coins = {coins} -> {a} dãy có thứ tự")

amount = 3, coins = [1, 2] -> 3 dãy có thứ tự
amount = 4, coins = [1, 2, 3] -> 7 dãy có thứ tự
amount = 7, coins = [2, 3] -> 3 dãy có thứ tự


### 15.3. Mỗi đồng xu dùng tối đa một lần (túi 0/1)

Chỉ cần đổi hướng duyệt `j` thành **giảm dần** (đúng "lỗi 3" ở trên nhưng giờ là yêu cầu). Kiểm bằng vét cạn các tập con.

In [32]:
def change_each_once(amount, coins):
    f = [1] + [0] * amount
    for x in coins:
        for j in range(amount, x - 1, -1):
            f[j] += f[j - x]
    return f[amount]

def change_each_once_bruteforce(amount, coins):
    m = len(coins)
    return sum(1 for mask in range(1 << m)
               if sum(coins[i] for i in range(m) if (mask >> i) & 1) == amount)

for amount, coins in [(5, [1, 2, 5]), (7, [1, 2, 3, 4, 5]), (0, [3]), (10, [2, 3, 5, 7])]:
    a = change_each_once(amount, coins)
    assert a == change_each_once_bruteforce(amount, coins)
    print(f"amount = {amount}, coins = {coins} -> {a} tập con")

amount = 5, coins = [1, 2, 5] -> 1 tập con
amount = 7, coins = [1, 2, 3, 4, 5] -> 3 tập con
amount = 0, coins = [3] -> 1 tập con
amount = 10, coins = [2, 3, 5, 7] -> 2 tập con


### 15.4. Liệt kê chính các tổ hợp (backtracking, LeetCode 39)

Khi cần **in** các tổ hợp (không chỉ đếm), ta quay lại backtracking; số tổ hợp phải khớp với DP.

In [33]:
def all_combinations(amount, coins):
    res = []
    def go(start, remain, path):
        if remain == 0:
            res.append(list(path))
            return
        for i in range(start, len(coins)):
            if coins[i] <= remain:
                path.append(coins[i])
                go(i, remain - coins[i], path)       # i, không phải i + 1: được dùng lại cùng xu
                path.pop()
    go(0, amount, [])
    return res

res = all_combinations(5, [1, 2, 5])
print(res)
assert len(res) == Solution().change(5, [1, 2, 5])
for amount, coins in [(10, [2, 3, 5]), (8, [1, 3, 4])]:
    assert len(all_combinations(amount, coins)) == Solution().change(amount, coins)
print("số tổ hợp liệt kê = số đếm bằng DP: OK")

[[1, 1, 1, 1, 1], [1, 1, 1, 2], [1, 2, 2], [5]]
số tổ hợp liệt kê = số đếm bằng DP: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | `product` mọi bộ số lượng, kiểm tổng | mũ theo `m` | `O(m)` | Chỉ để làm oracle (input rất nhỏ) |
| 2. Memo top-down | `dfs(i, remain)` = bỏ loại `i` + lấy thêm loại `i` | `O(m · n)` | `O(m · n)` + ngăn xếp sâu | Dễ nghĩ ra, nhưng coi chừng đệ quy sâu |
| 3. Bảng 2 chiều | `f[i][j] = f[i-1][j] + f[i][j-x]` | `O(m · n)` | `O(m · n)` | Học ý nghĩa từng ô, cần truy vết |
| 4. Một mảng | `f[j] += f[j-x]`, `j` tăng dần | `O(m · n)` | `O(n)` | **Nộp bài** |
| 5. Nhân đa thức | nhân `1 + x^c + x^2c + ...` | `O(Σ n²/c)` | `O(n)` | Góc nhìn đại số, kiểm chứng chéo |

Đo thời gian thực tế (số liệu chỉ để so sánh tương đối).

In [34]:
rng = random.Random(3)
amount, coins = 3000, rng.sample(range(1, 3000), 150)
ref = None
for name in ["memo", "table", "rolling"]:
    t0 = time.perf_counter()
    r = methods_dp[name](amount, coins)
    dt = time.perf_counter() - t0
    ref = r if ref is None else ref
    assert r == ref
    print(f"{name:8s} amount = {amount}, m = {len(coins)} -> {dt:.3f}s")

amount, coins = 400, rng.sample(range(1, 100), 20)
for name in ["rolling", "poly"]:
    t0 = time.perf_counter()
    r = methods_small[name](amount, coins)
    dt = time.perf_counter() - t0
    print(f"{name:8s} amount = {amount}, m = {len(coins)} -> {dt:.3f}s")

memo     amount = 3000, m = 150 -> 0.247s
table    amount = 3000, m = 150 -> 0.059s
rolling  amount = 3000, m = 150 -> 0.021s
rolling  amount = 400, m = 20 -> 0.001s
poly     amount = 400, m = 20 -> 0.006s


## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm, không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, dự đoán rồi đối chiếu đáp án ở mục 19:

```python
Solution().change(4, [1, 2, 3])
Solution().change(0, [5])
Solution().change(7, [2, 4])
Solution().change(6, [2, 3, 6])
```

### Bài 2 (tính tay)

Với `amount = 6`, `coins = [1, 3, 4]` hãy điền bảng `f[i][j]` (4 hàng × 7 cột) bằng tay theo mục 6 và liệt kê các tổ hợp tương ứng với `f[3][6]`.

### Bài 3 (điền TODO)

Hoàn thành hàm nén một mảng (hai chỗ `TODO`: giới hạn của vòng `j` và công thức cập nhật).

In [35]:
def change_fill_todo(amount, coins):
    f = [1] + [0] * amount
    for x in coins:
        # TODO: vòng lặp j chạy từ đâu đến đâu, theo hướng nào?
        # TODO: công thức cập nhật f[j]
        pass
    return f[amount]
# Không gọi change_fill_todo ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết hàm `change_memo(amount, coins)` bằng đệ quy có `lru_cache` theo trạng thái `(i, remain)` (cách 2).

In [36]:
def change_memo(amount, coins):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

**Số lượng xu giới hạn:** mỗi loại xu `coins[i]` chỉ có sẵn `cnt[i]` đồng. Viết `change_bounded(amount, coins, cnt)` đếm số tổ hợp tạo `amount`. Kiểm chứng với vét cạn `product`. (Gợi ý: với mỗi loại xu, hệ số mới `new[j] = Σ_{k=0..cnt} old[j - k·x]`.)

In [37]:
def change_bounded(amount, coins, cnt):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: liệt kê tay các tổ hợp của `4` bằng `1, 2, 3`: nhớ không đếm thứ tự.
- Mức 2: `amount = 0` luôn có đúng một cách (không lấy đồng nào); `7` lẻ mà chỉ có xu chẵn thì sao?
- Mức 3: `6` với `[2,3,6]` có `2+2+2`, `3+3`, `6`.

**Bài 2**

- Mức 1: hàng 0 chỉ có `f[0][0] = 1`; hàng 1 (xu 1) toàn `1`.
- Mức 2: hàng 2 (thêm xu 3): `f[2][j] = f[1][j] + f[2][j-3]`: `1,1,1,2,2,2,3`.
- Mức 3: hàng 3 thêm xu 4: `f[3][j] = f[2][j] + f[3][j-4]`; các tổ hợp của `6`: `1×6`, `3+1+1+1`, `3+3`, `4+1+1`.

**Bài 3**

- Mức 1: nhớ loại xu ở vòng ngoài, tổng ở vòng trong.
- Mức 2: `j` bắt đầu từ `x` (để khỏi chỉ số âm) và đi **tăng dần** tới `amount`.
- Mức 3: `for j in range(x, amount + 1): f[j] += f[j - x]`.

**Bài 4**

- Mức 1: hai nhánh: "bỏ loại `i`" và "lấy thêm một đồng loại `i`".
- Mức 2: điều kiện dừng: `remain == 0 → 1`; `i == len(coins) → 0`.
- Mức 3: `ways = dfs(i + 1, remain)`; nếu `remain >= coins[i]` thì `ways += dfs(i, remain - coins[i])`.

**Bài 5**

- Mức 1: khi mỗi loại có `cnt` đồng, số đồng dùng của loại đó là `k ∈ [0, cnt]`.
- Mức 2: duyệt từng loại, tạo mảng mới `new` từ `old`: `new[j] = old[j] + old[j - x] + ... + old[j - cnt·x]` (miễn chỉ số `>= 0`).
- Mức 3: ba vòng lặp: loại xu → tổng `j` → `k` từ `0` đến `min(cnt, j // x)`; so với `product(range(cnt+1), ...)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [38]:
# Bài 1
assert Solution().change(4, [1, 2, 3]) == 4
assert Solution().change(0, [5]) == 1
assert Solution().change(7, [2, 4]) == 0
assert Solution().change(6, [2, 3, 6]) == 3
print("Bài 1: 4, 1, 0, 3")

# Bài 2: bảng tay cho amount = 6, coins = [1, 3, 4]
amount, coins = 6, [1, 3, 4]
f = [[0] * (amount + 1) for _ in range(len(coins) + 1)]
f[0][0] = 1
for i, x in enumerate(coins, 1):
    for j in range(amount + 1):
        f[i][j] = f[i - 1][j] + (f[i][j - x] if j >= x else 0)
for i, row in enumerate(f):
    print(f"hàng {i}:", row)
assert f[3][6] == 4 and f[2] == [1, 1, 1, 2, 2, 2, 3]
print("các tổ hợp của 6:", all_combinations(6, [1, 3, 4]))

Bài 1: 4, 1, 0, 3
hàng 0: [1, 0, 0, 0, 0, 0, 0]
hàng 1: [1, 1, 1, 1, 1, 1, 1]
hàng 2: [1, 1, 1, 2, 2, 2, 3]
hàng 3: [1, 1, 1, 2, 3, 3, 4]
các tổ hợp của 6: [[1, 1, 1, 1, 1, 1], [1, 1, 1, 3], [1, 1, 4], [3, 3]]


In [39]:
# Bài 3
def change_fill_answer(amount, coins):
    f = [1] + [0] * amount
    for x in coins:
        for j in range(x, amount + 1):
            f[j] += f[j - x]
    return f[amount]

# Bài 4
def change_memo_answer(amount, coins):
    m = len(coins)
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))

    @lru_cache(maxsize=None)
    def dfs(i, remain):
        if remain == 0:
            return 1
        if i == m:
            return 0
        ways = dfs(i + 1, remain)
        if remain >= coins[i]:
            ways += dfs(i, remain - coins[i])
        return ways

    return dfs(0, amount)

# Bài 5
def change_bounded_answer(amount, coins, cnt):
    f = [1] + [0] * amount
    for x, c in zip(coins, cnt):
        new = [0] * (amount + 1)
        for j in range(amount + 1):
            for k in range(0, min(c, j // x) + 1):
                new[j] += f[j - k * x]
        f = new
    return f[amount]

def change_bounded_bruteforce(amount, coins, cnt):
    ranges = [range(c + 1) for c in cnt]
    return sum(1 for ks in product(*ranges) if sum(k * x for k, x in zip(ks, coins)) == amount)

rng = random.Random(1818)
for _ in range(200):
    m = rng.randint(1, 4)
    coins = rng.sample(range(1, 9), m)
    cnt = [rng.randint(0, 3) for _ in range(m)]
    amount = rng.randint(0, 20)
    assert change_fill_answer(amount, coins) == Solution().change(amount, coins)
    assert change_memo_answer(amount, coins) == Solution().change(amount, coins)
    assert change_bounded_answer(amount, coins, cnt) == change_bounded_bruteforce(amount, coins, cnt), (amount, coins, cnt)
print("Bài 3, 4, 5: OK")
print("ví dụ bounded: amount = 5, coins = [1, 2, 5], cnt = [2, 1, 1] ->", change_bounded_answer(5, [1, 2, 5], [2, 1, 1]))

Bài 3, 4, 5: OK
ví dụ bounded: amount = 5, coins = [1, 2, 5], cnt = [2, 1, 1] -> 1


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (cái túi không giới hạn, đếm tổ hợp):**

```python
class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        f = [1] + [0] * amount            # f[j]: số tổ hợp tạo tổng j; f[0] = 1 (tổ hợp rỗng)
        for x in coins:                    # vòng ngoài: LOẠI XU  -> đếm tổ hợp, không đếm thứ tự
            for j in range(x, amount + 1): # vòng trong: TỔNG, TĂNG dần -> dùng lại xu x nhiều lần
                f[j] += f[j - x]
        return f[amount]
```

**Câu thần chú:** *"Xu ở ngoài, tổng ở trong, đi lên; `f[0] = 1`."* (đảo ngoài/trong → hoán vị; đi xuống → mỗi xu một lần).

**Checklist trước khi submit:**

- [ ] Đề hỏi **số tổ hợp** (không thứ tự), mỗi xu dùng **vô hạn** lần.
- [ ] `f[0] = 1` (và `amount = 0` trả `1`).
- [ ] Vòng ngoài là **loại xu**, vòng trong là **tổng** chạy **tăng dần** từ `x`.
- [ ] Không có chỉ số âm lọt vào (`j >= x` hoặc `range(x, ...)`).
- [ ] Đã thử: `amount = 0`; xu lớn hơn `amount`; chỉ một loại xu; `coins` có `1`.
- [ ] Nếu dùng đệ quy: lưu ý độ sâu tới `amount / min(coins)` (có thể cần nâng giới hạn đệ quy, hoặc dùng bottom-up).
- [ ] Độ phức tạp `O(m · amount)` thời gian, `O(amount)` bộ nhớ: đủ cho `300 × 5000`.
- [ ] Đã đối chiếu bằng vét cạn trên input nhỏ trước khi nộp.